# 2.7 — Data Preparation for Health Gain Regression

## Objective
Predict **Health Gain** (OKS Post − Pre Score) as a **continuous regression target**.

## Design Rules

| Rule | Detail |
|------|--------|
| **Drop all Post-Op columns** | Any column with "Post" in the name leaks the outcome |
| **Regression target** | `Health Gain` — continuous, not a binary label |
| **No class balancing** | Not applicable for regression |
| **Benefit Label dropped at the end** | Kept only for audit steps, removed before saving |

## Steps

1. Load enriched dataset from `1.3-Health-Gain.parquet`
2. Drop unlabelled rows (null Health Gain)
3. Drop ALL Post-Op columns (leakage)
4. Drop identifier / constant columns
5. Drop predicted columns (already covered by step 3)
6. Audit remaining nulls
7. Impute nulls
8. Encode categoricals (Age Band, Gender, Year, Revision Flag OHE)
9. Class balancing — not applicable
10. Train / test split 80 / 20 + drop Benefit Label
11. Save + show all columns for verification

In [13]:
import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path

# ┌──────────────────────────────────────────────────────────────────────────────┐
# │  MODE — select which pipeline(s) to run                                      │
# │  "classification"  → predict Benefit Label (0/1)                            │
# │                       saves: 2.7-cls-train.parquet, 2.7-cls-test.parquet    │
# │  "regression"      → predict Health Gain (continuous)                       │
# │                       saves: 2.7-reg-train.parquet, 2.7-reg-test.parquet    │
# │  "both"            → run both pipelines, all four files saved               │
# └──────────────────────────────────────────────────────────────────────────────┘
MODE = "both"   # ← change this

RANDOM_SEED     = 42
HEALTH_GAIN_COL = "Health Gain"     # continuous regression target
LABEL_COL       = "Benefit Label"   # binary classification target (0=Benefit, 1=No Benefit)

assert MODE in ("classification", "regression", "both"), f"Invalid MODE: {MODE!r}"

Path("./data/interim").mkdir(parents=True, exist_ok=True)

print(f"Mode    : {MODE}")
print("Ready.")

Mode    : both
Ready.


---
## Step 1 — Load Data

Source: `1.3-Health-Gain.parquet` — 139,236 rows, 84 columns  
This file has the `Benefit Label` and `Health Gain` columns added by notebook 1.3.

In [14]:
df = pl.read_parquet("./data/interim/1.3-Health-Gain.parquet")

print(f"Rows   : {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print()

hg = df[HEALTH_GAIN_COL].drop_nulls()
print(f"Health Gain — valid rows : {len(hg):,}")
print(f"Health Gain — null rows  : {df[HEALTH_GAIN_COL].null_count():,}")
print(f"Health Gain — mean       : {hg.mean():.2f}")
print(f"Health Gain — median     : {hg.median():.2f}")
print(f"Health Gain — min/max    : [{hg.min()}, {hg.max()}]")
print()
print(f"Benefit Label — 0 (Benefit)   : {int((df[LABEL_COL] == 0).sum()):,}")
print(f"Benefit Label — 1 (No Benefit): {int((df[LABEL_COL] == 1).sum()):,}")
print(f"Benefit Label — null          : {df[LABEL_COL].null_count():,}")

Rows   : 139,236
Columns: 84

Health Gain — valid rows : 135,051
Health Gain — null rows  : 4,185
Health Gain — mean       : 16.89
Health Gain — median     : 17.00
Health Gain — min/max    : [-37.0, 47.0]

Benefit Label — 0 (Benefit)   : 111,953
Benefit Label — 1 (No Benefit): 23,098
Benefit Label — null          : 4,185


---
## Step 2 — Drop Unlabelled Rows

4,185 rows have a null `Health Gain` — these are patients where either the Pre-Op or Post-Op OKS score was missing.  
No No-Benefit patients exist in these null rows (verified in notebook 1.3), so dropping them is safe.

In [15]:
before = df.shape[0]

df = df.filter(pl.col(HEALTH_GAIN_COL).is_not_null())

dropped = before - df.shape[0]
print(f"Rows before : {before:,}")
print(f"Rows dropped: {dropped:,}  (null Health Gain — Pre or Post OKS score missing)")
print(f"Rows after  : {df.shape[0]:,}")
print(f"No Benefit rows retained: {int((df[LABEL_COL] == 1).sum()):,}  ✓")

Rows before : 139,236
Rows dropped: 4,185  (null Health Gain — Pre or Post OKS score missing)
Rows after  : 135,051
No Benefit rows retained: 23,098  ✓


---
## Step 3 — Drop ALL Post-Op Columns (Leakage)

Any column containing **"Post"** in its name is measured **after surgery** and therefore leaks the outcome.  
Since `Health Gain = Post Score − Pre Score`, every post-op measurement directly or indirectly encodes the target.

All such columns are dropped with a single filter — no manual list needed.

In [16]:
post_cols = [c for c in df.columns if "Post" in c]

print(f"Columns containing 'Post' to drop ({len(post_cols)}):")
for c in post_cols:
    print(f"  {c}")

df = df.drop(post_cols)

print(f"\nColumns remaining: {df.shape[1]}")
print(f"Health Gain still present : {HEALTH_GAIN_COL in df.columns}  ✓")
print(f"Benefit Label still present: {LABEL_COL in df.columns}  ✓")
print(f"Pre-Op columns still present: {sum(1 for c in df.columns if 'Pre' in c)}")

Columns containing 'Post' to drop (36):
  Post-Op Q Assisted
  Post-Op Q Assisted By
  Post-Op Q Living Arrangements
  Post-Op Q Disability
  Post-Op Q Mobility
  Post-Op Q Self-Care
  Post-Op Q Activity
  Post-Op Q Discomfort
  Post-Op Q Anxiety
  Post-Op Q Satisfaction
  Post-Op Q Sucess
  Post-Op Q Allergy
  Post-Op Q Bleeding
  Post-Op Q Wound
  Post-Op Q Urine
  Post-Op Q Further Surgery
  Post-Op Q Readmitted
  Post-Op Q EQ5D Index Profile
  Post-Op Q EQ5D Index
  Knee Replacement EQ 5D Index Post-Op Q Predicted
  Post-Op Q EQ VAS
  Knee Replacement EQ VAS_Post-Op Q Predicted
  Knee Replacement Post-Op Q Pain
  Knee Replacement Post-Op Q Night Pain
  Knee Replacement Post-Op Q Washing
  Knee Replacement Post-Op Q Transport
  Knee Replacement Post-Op Q Walking
  Knee Replacement Post-Op Q Standing
  Knee Replacement Post-Op Q Limping
  Knee Replacement Post-Op Q Kneeling
  Knee Replacement Post-Op Q Work
  Knee Replacement Post-Op Q Confidence
  Knee Replacement Post-Op Q Shopping

---
## Step 4 — Drop Identifier / Constant Columns

| Column | Reason |
|--------|--------|
| `Provider Code` | Hospital identifier — not a patient feature; very high cardinality |
| `Procedure` | Constant value ("Knee Replacement") — zero information |
| `CSVYear` | Duplicate of `Year` (same data, different format) |

In [17]:
ID_CONST_COLS = ["Provider Code", "Procedure", "CSVYear"]
ID_CONST_COLS = [c for c in ID_CONST_COLS if c in df.columns]
df = df.drop(ID_CONST_COLS)

print(f"Dropped {len(ID_CONST_COLS)} identifier/constant columns.")
print(f"Columns remaining: {df.shape[1]}")

Dropped 3 identifier/constant columns.
Columns remaining: 45


---
## Step 5 — Drop Predicted / Model-Output Columns

Three columns store **model-predicted post-op values** that were pre-computed by the NHS and stored in the raw data.  
They are not real patient measurements and have the highest null rates (10–14%).  
They also contain No Benefit patients in their null rows, so we drop the **columns** rather than any rows.

| Column | Null % | No Benefit in nulls |
|--------|--------|---------------------|
| `Knee Replacement EQ VAS_Post-Op Q Predicted` | 13.9% | 3,554 |
| `Knee Replacement EQ 5D Index Post-Op Q Predicted` | 10.3% | 2,509 |
| `Knee Replacement OKS Post-Op Q Predicted` | 3.9% | 209 |

In [18]:
PREDICTED_COLS = [
    "Knee Replacement EQ VAS_Post-Op Q Predicted",
    "Knee Replacement EQ 5D Index Post-Op Q Predicted",
    "Knee Replacement OKS Post-Op Q Predicted",
]
PREDICTED_COLS = [c for c in PREDICTED_COLS if c in df.columns]
df = df.drop(PREDICTED_COLS)

print(f"Dropped {len(PREDICTED_COLS)} predicted columns (already removed by Step 3 if any).")
print(f"Columns remaining: {df.shape[1]}")
print(f"No Benefit rows  : {int((df[LABEL_COL] == 1).sum()):,}  ✓ (unchanged)")

Dropped 0 predicted columns (already removed by Step 3 if any).
Columns remaining: 45
No Benefit rows  : 23,098  ✓ (unchanged)


---
## Step 6 — Audit Remaining Nulls

For every column still containing nulls, we show:
- How many rows are null
- How many of those null rows are No Benefit patients

**Rule:** if a column has No Benefit patients in its null rows → we **impute**, not drop rows.

In [19]:
null_cols = [c for c in df.columns if df[c].null_count() > 0]

rows = []
for c in null_cols:
    total_null      = df[c].null_count()
    null_pct        = round(100 * total_null / df.shape[0], 1)
    nb_in_null      = int(df.filter(pl.col(c).is_null() & (pl.col(LABEL_COL) == 1)).shape[0])
    action          = "IMPUTE (preserves No Benefit)" if nb_in_null > 0 else "drop rows (0 No Benefit in nulls)"
    rows.append({
        "Column"              : c,
        "Null Count"          : total_null,
        "Null %"              : null_pct,
        "No Benefit in nulls" : nb_in_null,
        "Action"              : action,
    })

audit = pd.DataFrame(rows).sort_values("Null %", ascending=False)
print(f"Columns with remaining nulls: {len(audit)}")
print()
print(audit.to_string(index=False))

Columns with remaining nulls: 3

             Column  Null Count  Null %  No Benefit in nulls                        Action
           Age Band        9127     6.8                 1684 IMPUTE (preserves No Benefit)
             Gender        9127     6.8                 1684 IMPUTE (preserves No Benefit)
Pre-Op Q EQ5D Index        7127     5.3                 1317 IMPUTE (preserves No Benefit)


---
## Step 7 — Impute Remaining Nulls

All remaining null columns contain No Benefit patients in their null rows — we **must not** drop those rows.

| Column | Strategy | Reason |
|--------|----------|--------|
| `Age Band` | Fill with `"Unknown"` | Categorical; creating a new category is more honest than guessing age |
| `Gender` | Fill with `0` ("Unknown") | Maps to a new "not recorded" category |
| `Pre-Op Q EQ5D Index` | Fill with **median** | Continuous; median is robust to skew |

In [20]:
rows_before = df.shape[0]

# ── Age Band → fill null with "Unknown" ──────────────────────────────────────
if "Age Band" in df.columns:
    df = df.with_columns(
        pl.col("Age Band").cast(pl.Utf8).fill_null("Unknown").cast(pl.Categorical).alias("Age Band")
    )

# ── Gender → fill null with "0" (Unknown) ────────────────────────────────────
if "Gender" in df.columns:
    df = df.with_columns(
        pl.col("Gender").cast(pl.Utf8).fill_null("0").cast(pl.Categorical).alias("Gender")
    )

# ── Pre-Op Q EQ5D Index → fill null with overall median ──────────────────────
if "Pre-Op Q EQ5D Index" in df.columns:
    median_pre_eq5d = float(df["Pre-Op Q EQ5D Index"].drop_nulls().median())
    df = df.with_columns(pl.col("Pre-Op Q EQ5D Index").fill_null(median_pre_eq5d))
    print(f"Pre-Op Q EQ5D Index median fill: {median_pre_eq5d:.4f}")

# ── Verify ────────────────────────────────────────────────────────────────────
remaining_nulls = sum(df[c].null_count() for c in df.columns if c not in (LABEL_COL,))
print(f"\nRemaining nulls (excl. label): {remaining_nulls}")
print(f"Rows unchanged: {df.shape[0] == rows_before}  ✓  ({df.shape[0]:,} rows)")

Pre-Op Q EQ5D Index median fill: 0.5870

Remaining nulls (excl. label): 0
Rows unchanged: True  ✓  (135,051 rows)


---
## Step 8 — Encode Categorical Columns

Convert string / categorical columns to integers so they are ready for modelling.

| Column | Encoding | Values |
|--------|----------|--------|
| `Age Band` | Ordinal | Unknown=0, 40–49=1, 50–59=2, 60–69=3, 70–79=4, 80–89=5, 90–120=6 |
| `Gender` | Label | 0=Unknown, 1=Male, 2=Female |
| `Year` | Ordinal | 2016/17=0, 2017/18=1, 2018/19=2 |
| `Revision Flag` | One-hot | `Revision Flag_0` (primary), `Revision Flag_1` (revision surgery) |

In [21]:
AGE_MAP = {
    "Unknown":   0,
    "40 to 49":  1,
    "50 to 59":  2,
    "60 to 69":  3,
    "70 to 79":  4,
    "80 to 89":  5,
    "90 to 120": 6,
}

GENDER_MAP = {"0": 0, "1": 1, "2": 2}

YEAR_MAP = {"2016/17": 0, "2017/18": 1, "2018/19": 2}

encode_exprs = []

if "Age Band" in df.columns:
    encode_exprs.append(
        pl.col("Age Band").cast(pl.Utf8)
          .replace(AGE_MAP, default=0)
          .cast(pl.Int8)
          .alias("Age Band")
    )

if "Gender" in df.columns:
    encode_exprs.append(
        pl.col("Gender").cast(pl.Utf8)
          .replace(GENDER_MAP, default=0)
          .cast(pl.Int8)
          .alias("Gender")
    )

if "Year" in df.columns:
    encode_exprs.append(
        pl.col("Year").cast(pl.Utf8)
          .replace(YEAR_MAP, default=0)
          .cast(pl.Int8)
          .alias("Year")
    )

if encode_exprs:
    df = df.with_columns(encode_exprs)

# ── One-hot encode Revision Flag ─────────────────────────────────────────────
if "Revision Flag" in df.columns:
    df = df.with_columns(pl.col("Revision Flag").cast(pl.Int32))
    ohe = df.select(pl.col("Revision Flag")).to_dummies(columns=["Revision Flag"])
    # Cast OHE columns to Int8 (0/1 values only)
    ohe = ohe.with_columns([pl.col(c).cast(pl.Int8) for c in ohe.columns])
    df = df.drop("Revision Flag").hstack(ohe)

print("Encoding complete.")
print()
print("One-hot encoded Revision Flag columns:")
rf_cols = [c for c in df.columns if "Revision Flag" in c]
print(df.select(rf_cols).head(5))
print(f"\nValue counts:")
for c in rf_cols:
    print(f"  {c}: {df[c].value_counts().sort(c)}")

Encoding complete.

One-hot encoded Revision Flag columns:
shape: (5, 2)
┌─────────────────┬─────────────────┐
│ Revision Flag_0 ┆ Revision Flag_1 │
│ ---             ┆ ---             │
│ i8              ┆ i8              │
╞═════════════════╪═════════════════╡
│ 1               ┆ 0               │
│ 1               ┆ 0               │
│ 1               ┆ 0               │
│ 1               ┆ 0               │
│ 1               ┆ 0               │
└─────────────────┴─────────────────┘

Value counts:
  Revision Flag_0: shape: (2, 2)
┌─────────────────┬────────┐
│ Revision Flag_0 ┆ count  │
│ ---             ┆ ---    │
│ i8              ┆ u32    │
╞═════════════════╪════════╡
│ 0               ┆ 5068   │
│ 1               ┆ 129983 │
└─────────────────┴────────┘
  Revision Flag_1: shape: (2, 2)
┌─────────────────┬────────┐
│ Revision Flag_1 ┆ count  │
│ ---             ┆ ---    │
│ i8              ┆ u32    │
╞═════════════════╪════════╡
│ 0               ┆ 129983 │
│ 1               ┆ 50

C:\Users\mittall\AppData\Local\Temp\ipykernel_22376\1529688656.py:19: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("Age Band").cast(pl.Utf8)
C:\Users\mittall\AppData\Local\Temp\ipykernel_22376\1529688656.py:27: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("Gender").cast(pl.Utf8)
C:\Users\mittall\AppData\Local\Temp\ipykernel_22376\1529688656.py:35: DeprecationWarning: the `default` parameter for `replace` is deprecated. Use `replace_strict` instead to set a default while replacing values.
(Deprecated in version 1.0.0)
  pl.col("Year").cast(pl.Utf8)


---
## Step 9 — Balance Dataset

All modes use the same balanced dataset to ensure a fair model comparison.

**Strategy**: Keep **all No Benefit rows** (minority class), then randomly undersample **Benefit rows** to match.  
This gives a 1:1 Benefit / No Benefit ratio, so models are not biased by class imbalance.

> The balanced `df_balanced` is then split per MODE in Step 10.

In [22]:
no_benefit      = df.filter(pl.col(LABEL_COL) == 1)
benefit         = df.filter(pl.col(LABEL_COL) == 0)
n_no_benefit    = no_benefit.shape[0]

# Undersample Benefit to match No Benefit count
benefit_sampled = benefit.sample(n=n_no_benefit, shuffle=True, seed=RANDOM_SEED)

# Combine and shuffle
df_balanced = (
    pl.concat([no_benefit, benefit_sampled])
      .sample(fraction=1.0, shuffle=True, seed=RANDOM_SEED)
)

print(f"No Benefit rows (kept all)    : {n_no_benefit:,}")
print(f"Benefit rows (undersampled)   : {benefit_sampled.shape[0]:,}")
print(f"Benefit rows (original total) : {benefit.shape[0]:,}  — {benefit.shape[0] - benefit_sampled.shape[0]:,} removed")
print(f"Total balanced rows           : {df_balanced.shape[0]:,}")
print(f"Class ratio Benefit:NoBenefit = 1:1  ✓")

No Benefit rows (kept all)    : 23,098
Benefit rows (undersampled)   : 23,098
Benefit rows (original total) : 111,953  — 88,855 removed
Total balanced rows           : 46,196
Class ratio Benefit:NoBenefit = 1:1  ✓


---
## Step 10 — Train / Test Split (80 / 20) per MODE

Each mode gets its own train/test split from the same balanced dataset.  
The column that is **not** the target for that mode is dropped before splitting.

| MODE | Column kept | Column dropped |
|------|-------------|----------------|
| classification | `Benefit Label` | `Health Gain` |
| regression | `Health Gain` | `Benefit Label` |

In [23]:
TEST_RATIO = 0.20
results    = {}   # keys: "cls" and/or "reg"

def split_80_20(data: pl.DataFrame, seed: int = RANDOM_SEED):
    shuffled = data.sample(fraction=1.0, shuffle=True, seed=seed)
    n_test   = int(len(shuffled) * TEST_RATIO)
    return shuffled[n_test:], shuffled[:n_test]   # (train, test)

# ── Classification ────────────────────────────────────────────────────────────
if MODE in ("classification", "both"):
    df_cls            = df_balanced.drop(HEALTH_GAIN_COL)
    train_cls, test_cls = split_80_20(df_cls)
    results["cls"]    = (train_cls, test_cls)
    print(f"[cls] Train: {train_cls.shape[0]:,}  Test: {test_cls.shape[0]:,}")
    print(f"      Train — Benefit: {int((train_cls[LABEL_COL]==0).sum()):,}  No Benefit: {int((train_cls[LABEL_COL]==1).sum()):,}")
    print(f"      Test  — Benefit: {int((test_cls[LABEL_COL]==0).sum()):,}  No Benefit: {int((test_cls[LABEL_COL]==1).sum()):,}")
    print()

# ── Regression ────────────────────────────────────────────────────────────────
if MODE in ("regression", "both"):
    df_reg            = df_balanced.drop(LABEL_COL)
    train_reg, test_reg = split_80_20(df_reg)
    results["reg"]    = (train_reg, test_reg)
    print(f"[reg] Train: {train_reg.shape[0]:,}  Test: {test_reg.shape[0]:,}")
    print(f"      Train — Health Gain mean: {train_reg[HEALTH_GAIN_COL].mean():.2f}  std: {train_reg[HEALTH_GAIN_COL].std():.2f}")
    print(f"      Test  — Health Gain mean: {test_reg[HEALTH_GAIN_COL].mean():.2f}  std: {test_reg[HEALTH_GAIN_COL].std():.2f}")

[cls] Train: 36,957  Test: 9,239
      Train — Benefit: 18,507  No Benefit: 18,450
      Test  — Benefit: 4,591  No Benefit: 4,648

[reg] Train: 36,957  Test: 9,239
      Train — Health Gain mean: 10.84  std: 11.20
      Test  — Health Gain mean: 10.71  std: 11.25


---
## Step 11 — Final Summary & Save

A pipeline summary table confirms every constraint was met before saving.

In [24]:
def verify_and_save(train, test, prefix, target_col, forbidden_col, save_as=None):
    print(f"\n  Train shape: {train.shape}  |  Test shape: {test.shape}")

    checks = [
        (f"Target '{target_col}' in train",  target_col in train.columns,                           True),
        (f"'{forbidden_col}' removed",        forbidden_col not in train.columns,                   True),
        ("No Post-Op columns",               sum(1 for c in train.columns if "Post" in c),           0),
        ("Nulls in train",                   sum(train[c].null_count() for c in train.columns),      0),
        ("Nulls in test",                    sum(test[c].null_count()  for c in test.columns),        0),
    ]
    all_pass = True
    for name, actual, expected in checks:
        ok = (actual == expected)
        all_pass = all_pass and ok
        print(f"  [{'PASS' if ok else 'FAIL'}]  {name}: {actual}")

    # Feature column list for manual review
    feature_cols = [c for c in train.columns if c != target_col]
    print(f"\n  Feature columns ({len(feature_cols)}):")
    for i, c in enumerate(feature_cols, 1):
        dtype = str(train[c].dtype)
        nulls = train[c].null_count()
        print(f"    {i:>3}. {c:<55}  dtype={dtype:<12}  nulls={nulls}")

    if all_pass:
        # ── Rename target column to match what the modelling notebook expects ─
        if save_as and save_as != target_col:
            train = train.rename({target_col: save_as})
            test  = test.rename({target_col:  save_as})
            print(f"\n  Renamed: '{target_col}'  →  '{save_as}'")

        train_path = f"./data/interim/{prefix}-train.parquet"
        test_path  = f"./data/interim/{prefix}-test.parquet"
        train.write_parquet(train_path, compression="gzip")
        test.write_parquet(test_path,  compression="gzip")
        print(f"  Saved: {train_path}")
        print(f"  Saved: {test_path}")
    else:
        print("\n  ERROR: One or more checks failed — files NOT saved.")

    return all_pass


print("=" * 65)
print(f"PIPELINE 2.7 — MODE={MODE} — FINAL SUMMARY")
print("=" * 65)

if "cls" in results:
    print("\n── CLASSIFICATION ──────────────────────────────────────────")
    # 3.1-Classification-Modelling.ipynb expects TARGET = 'NO_Benefit'
    verify_and_save(*results["cls"], "2.7-cls", LABEL_COL, HEALTH_GAIN_COL, save_as="NO_Benefit")

if "reg" in results:
    print("\n── REGRESSION ──────────────────────────────────────────────")
    # 3.2-Data-Modelling-Optimised.ipynb expects TARGET = 'health_gain'
    verify_and_save(*results["reg"], "2.7-reg", HEALTH_GAIN_COL, LABEL_COL, save_as="health_gain")

PIPELINE 2.7 — MODE=both — FINAL SUMMARY

── CLASSIFICATION ──────────────────────────────────────────

  Train shape: (36957, 45)  |  Test shape: (9239, 45)
  [PASS]  Target 'Benefit Label' in train: True
  [PASS]  'Health Gain' removed: True
  [PASS]  No Post-Op columns: 0
  [PASS]  Nulls in train: 0
  [PASS]  Nulls in test: 0

  Feature columns (44):
      1. Year                                                     dtype=Int8          nulls=0
      2. Age Band                                                 dtype=Int8          nulls=0
      3. Gender                                                   dtype=Int8          nulls=0
      4. Pre-Op Q Assisted                                        dtype=UInt8         nulls=0
      5. Pre-Op Q Assisted By                                     dtype=UInt8         nulls=0
      6. Pre-Op Q Symptom Period                                  dtype=UInt8         nulls=0
      7. Pre-Op Q Previous Surgery                                dtype=UInt8   